# PhishGuard — Phishing URL Detection Model

Trains a classifier to detect phishing URLs from 15 engineered features and
exports the winning model to ONNX for on-device Android (Java) inference.

**Label convention:** `0 = legitimate`, `1 = phishing`
**Target:** F1 >= 0.85 on a 20% held-out test set.

Steps: load/inspect -> clean -> extract 15 features -> scale -> train 6 models ->
evaluate -> GridSearchCV -> final report -> save artifacts -> ONNX export -> verify.


## Step 0 — Imports & config

In [ ]:
import os, json, glob, warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
import joblib
from xgboost import XGBClassifier

from feature_extractor import extract_features, FEATURE_NAMES

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)
print("Features:", FEATURE_NAMES)


## Step 1 — Load dataset & inspect

Prefers the real **PhiUSIIL** dataset if present (raw URLs + label). Note its
label convention is inverted vs. ours, so we flip it to `0=legit, 1=phish`.
Falls back to the local archive CSV or the synthetic dataset.

In [ ]:
def load_raw_dataframe():
    phiusiil = glob.glob(os.path.join("..", "..", "phiusiil+phishing+url+dataset", "*.csv"))
    if phiusiil:
        print("Using PhiUSIIL:", phiusiil[0])
        df = pd.read_csv(phiusiil[0], usecols=["URL", "label"]).rename(columns={"URL": "url"})
        df["label"] = 1 - df["label"].astype(int)   # flip to our convention
        return df[["url", "label"]]
    archive = os.path.join("..", "..", "archive", "phishing_features.csv")
    if os.path.exists(archive):
        print("Using archive:", archive)
        return pd.read_csv(archive, usecols=["url", "label"])[["url", "label"]]
    print("Using synthetic data")
    return pd.read_csv(os.path.join("data", "phishing_urls.csv"))[["url", "label"]]

df = load_raw_dataframe()
print("Shape:", df.shape)
print("Missing:\n", df.isnull().sum())
print("Class distribution:\n", df["label"].value_counts())
df.head()


## Step 2 — Clean: drop nulls, remove duplicates, fill numeric nulls with median

In [ ]:
before = len(df)
df = df.dropna(subset=["url", "label"]).drop_duplicates(subset=["url"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)
print(f"Removed {before - len(df)} rows -> {len(df)} remain")

# Cap large datasets for speed while staying well above 5,000 rows.
MAX_ROWS = 40000
if len(df) > MAX_ROWS:
    df, _ = train_test_split(df, train_size=MAX_ROWS, random_state=RANDOM_STATE, stratify=df["label"])
    df = df.reset_index(drop=True)
print("Rows used:", len(df))


## Step 3 — Extract the 15 features and split 80/20 (stratified)

In [ ]:
X = pd.DataFrame([extract_features(u) for u in df["url"]], columns=FEATURE_NAMES)
y = df["label"].values
X = X.fillna(X.median(numeric_only=True))   # fill numeric nulls with median

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)
X.head()


## Step 4 — StandardScaler (fit on train, transform on test)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Step 5 & 6 — Train 6 models and compare

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                             subsample=0.9, colsample_bytree=0.9,
                             eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1),
    "GaussianNB": GaussianNB(),
    "SVC": SVC(probability=True, random_state=RANDOM_STATE),
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    results.append((name,
                    accuracy_score(y_test, pred),
                    precision_score(y_test, pred, zero_division=0),
                    recall_score(y_test, pred, zero_division=0),
                    f1_score(y_test, pred, zero_division=0)))
    print(name, "confusion matrix:\n", confusion_matrix(y_test, pred))

comparison = pd.DataFrame(results, columns=["Model","Accuracy","Precision","Recall","F1"]
                          ).sort_values("F1", ascending=False).reset_index(drop=True)
comparison


## Step 7 — GridSearchCV on the best model (cv=5, scoring='f1')

In [ ]:
best_name = comparison.iloc[0]["Model"]
print("Best base model:", best_name)

param_grids = {
    "RandomForest": {"n_estimators":[100,200,300], "max_depth":[None,10,20], "min_samples_split":[2,5]},
    "XGBoost": {"n_estimators":[100,200,300], "max_depth":[4,6,8], "learning_rate":[0.05,0.1,0.2]},
    "DecisionTree": {"max_depth":[None,10,20,30], "min_samples_split":[2,5,10], "criterion":["gini","entropy"]},
    "LogisticRegression": {"C":[0.01,0.1,1,10], "penalty":["l2"], "solver":["lbfgs"]},
    "SVC": {"C":[0.1,1,10], "kernel":["rbf","linear"], "gamma":["scale"]},
    "GaussianNB": {"var_smoothing":[1e-9,1e-8,1e-7,1e-6]},
}
base = {
    "RandomForest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1),
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "SVC": SVC(probability=True, random_state=RANDOM_STATE),
    "GaussianNB": GaussianNB(),
}
grid = GridSearchCV(base[best_name], param_grids[best_name], cv=5, scoring="f1", n_jobs=-1, verbose=1)
grid.fit(X_train_scaled, y_train)
print("Best params:", grid.best_params_, "CV f1:", round(grid.best_score_, 4))
best_model = grid.best_estimator_


## Step 8 — Final evaluation on the test set

In [ ]:
final_pred = best_model.predict(X_test_scaled)
final_f1 = f1_score(y_test, final_pred, zero_division=0)
print(classification_report(y_test, final_pred, target_names=["legit(0)","phish(1)"]))
print("Confusion matrix:\n", confusion_matrix(y_test, final_pred))
print(f"FINAL TEST F1 = {final_f1:.4f} (target >= 0.85)")


## Step 9 — Refit on RAW features & save artifacts

The Android app feeds **raw** `float[15]` (no on-device scaler). Tree ensembles
are scale-invariant, so we deploy a model trained on raw features. This makes
the ONNX input contract exactly match what the app produces.

In [ ]:
deploy_model = clone(best_model)
deploy_model.fit(X_train.values.astype(np.float32), y_train)
raw_f1 = f1_score(y_test, deploy_model.predict(X_test.values.astype(np.float32)), zero_division=0)
print("Raw-feature deploy model test F1 =", round(raw_f1, 4))

joblib.dump(deploy_model, os.path.join(MODELS_DIR, "optimized_model.pkl"))
joblib.dump(scaler, os.path.join(MODELS_DIR, "scaler.pkl"))
with open(os.path.join(MODELS_DIR, "feature_names.json"), "w") as f:
    json.dump(FEATURE_NAMES, f, indent=2)
print("Saved optimized_model.pkl, scaler.pkl, feature_names.json")


## Step 10 — Convert to ONNX (`float_input`, shape `[None, 15]`)

In [ ]:
onnx_path = os.path.join(MODELS_DIR, "phishing_model.onnx")
if isinstance(deploy_model, XGBClassifier):
    from onnxmltools.convert import convert_xgboost
    from onnxmltools.convert.common.data_types import FloatTensorType
    onnx_model = convert_xgboost(deploy_model, initial_types=[("float_input", FloatTensorType([None, 15]))])
else:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    onnx_model = convert_sklearn(deploy_model,
                                 initial_types=[("float_input", FloatTensorType([None, 15]))],
                                 options={id(deploy_model): {"zipmap": False}})
with open(onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())
print("Saved", onnx_path)


## Step 11 — Verify ONNX vs sklearn on 5 raw test samples

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name
print("ONNX input:", input_name, "| outputs:", [o.name for o in sess.get_outputs()])

sample = X_test.iloc[:5].values.astype(np.float32)
sk = deploy_model.predict(sample)
ox = np.asarray(sess.run(None, {input_name: sample})[0]).ravel()
for i in range(5):
    print(f"sample {i}: sklearn={int(sk[i])} onnx={int(ox[i])} match={int(sk[i])==int(ox[i])}")
print("ALL MATCH" if all(int(sk[i])==int(ox[i]) for i in range(5)) else "MISMATCH")
